# Bomberman: Shielded Linear PPO

This notebook tests how much the agent improves from PPO training compared with the handcrafted starting policy and the safety mask.

The V4 agent uses the same features, initial weights and four-stage curriculum: 20 navigation rounds, 70 bomb-escape rounds, 60 mixed-opponent rounds and 150 combat rounds. The experiments compare the initial policy, the 90-round checkpoint, the 300-round checkpoint and a basic-mask version of the 300-round policy.

Run the cells in order in a CPU Colab runtime. Use `profile="quick"` for a short integration check and `profile="ppo300"` for the full experiment. Existing checkpoints can be evaluated with `mode="evaluate_existing"`.


## 0. Experiment overview

| Comparison | Purpose | Kept the same |
|---|---|---|
| Initial policy vs 90-round PPO | Effect of PPO updates | Full mask, features, greedy inference, opponents and seed blocks |
| 90-round vs 300-round PPO | Effect of the later training stages | Evaluation setup and full mask |
| 300-round PPO: full vs basic mask | Effect of the full action mask at inference | Actor/critic weights and feature computation |

The basic mask keeps physical legality and immediate blast checks, but removes the longer-horizon escape and bomb-safety checks used by the full mask. The policy weights are unchanged for this comparison.


## 1. Evaluation setup

1. Train the agent once, or load the saved 90-round and 300-round checkpoints. The untrained handcrafted policy is saved separately.
2. Compare the two trained checkpoints on **20 development seeds × 5 rounds = 100 games per checkpoint**.
3. Evaluate the initial policy, both PPO checkpoints and the 300-round basic-mask variant on a separate set of **20 test seeds × 5 rounds = 100 games per variant**.
4. Use matched seed blocks for paired differences in score, survival, suicide rate and kills, with 95% t intervals.

All variants use the same scenario, three seeded rule-based opponents and greedy action selection. Development and test seed sets are separate. The historical V3 score of 3.864 is included only as a reference value because it was not measured with the same matched setup.


## 2. Colab setup

The agent and PPO implementation use NumPy on the CPU, so a GPU is not required.


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/ukoethe/bomberman_rl.git"
REPO_DIR = Path("/content/bomberman_rl")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Using existing checkout without pulling; commit is recorded in the run manifest.")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pygame", "tqdm", "pandas", "matplotlib", "seaborn", "scikit-learn", "scipy", "joblib",
], check=True)
os.environ.update({
    "SDL_VIDEODRIVER": "dummy",
    "SDL_AUDIODRIVER": "dummy",
    "PYGAME_HIDE_SUPPORT_PROMPT": "1",
})
os.chdir(REPO_DIR)
print("Framework ready at", REPO_DIR)

## 3. Configuration

The full profile runs 300 training games, 200 development games and 400 test games, plus a short smoke test. Runtime depends on episode length and Colab CPU load.

- `mode="fresh"`: start a new training run from the saved initial policy.
- `mode="resume"`: continue from a previous run directory.
- `mode="evaluate_existing"`: load the 90-round and 300-round checkpoints and skip training.

`profile="quick"` uses a much smaller run to check that the pipeline works. Final export is disabled in quick mode. Evaluation results are cached only when the checkpoint, source code, opponents and protocol still match.


In [ ]:
from dataclasses import asdict, dataclass

@dataclass(frozen=True)
class ExperimentConfig:
    profile: str = "ppo300"                 # "quick" is integration-only
    mode: str = "fresh"                     # fresh / resume / evaluate_existing
    resume_run_dir: str = ""
    checkpoint_90: str = ""                 # e.g. /content/01_ppo_bomb_escape.pkl
    checkpoint_300: str = ""                # e.g. /content/03_ppo_combat.pkl
    base_seed: int = 7300
    dev_seeds: tuple = tuple(range(24101, 24121))
    dev_rounds_per_seed: int = 5
    final_seeds: tuple = tuple(range(35101, 35121))
    final_rounds_per_seed: int = 5
    v3_score_baseline: float = 3.864
    max_suicide_rate: float = 0.25
    max_invalid_rate: float = 0.01
    max_mean_act_ms: float = 100.0

CONFIG = ExperimentConfig()
assert CONFIG.profile in {"ppo300", "quick"}
assert CONFIG.mode in {"fresh", "resume", "evaluate_existing"}
if CONFIG.mode == "evaluate_existing" and CONFIG.profile == "quick":
    raise ValueError("Use ppo300 to import the actual 90/300-round checkpoints.")
OLD_SEEDS = {4101, 4102, 4103, 17, 29, 43, 71, 97,
             131, 149, 167, 181, 199, 223, 241, 263, 281, 307}
TRAIN_SEEDS = {CONFIG.base_seed + i for i in range(1, 5)} | {991}
DEV_SEEDS = CONFIG.dev_seeds[:2] if CONFIG.profile == "quick" else CONFIG.dev_seeds
TEST_SEEDS = CONFIG.final_seeds[:2] if CONFIG.profile == "quick" else CONFIG.final_seeds
DEV_ROUNDS = 1 if CONFIG.profile == "quick" else CONFIG.dev_rounds_per_seed
TEST_ROUNDS = 1 if CONFIG.profile == "quick" else CONFIG.final_rounds_per_seed
assert len(DEV_SEEDS) >= 2 and len(TEST_SEEDS) >= 2
assert len(set(DEV_SEEDS)) == len(DEV_SEEDS) and len(set(TEST_SEEDS)) == len(TEST_SEEDS)
assert not set(DEV_SEEDS) & set(TEST_SEEDS)
assert not (set(DEV_SEEDS) | set(TEST_SEEDS)) & (OLD_SEEDS | TRAIN_SEEDS)
assert DEV_ROUNDS > 0 and TEST_ROUNDS > 0
PPO_AGENT = "survival_linear_ppo_v4"
EARLY = "quick_early_full" if CONFIG.profile == "quick" else "ppo_90_full"
LATE = "quick_late_full" if CONFIG.profile == "quick" else "ppo_300_full"
AGENT_DIRS = {PPO_AGENT: REPO_DIR / "agent_code" / PPO_AGENT}
for directory in AGENT_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)
    (directory / "__init__.py").touch()
print(asdict(CONFIG))
print("Evaluation games:", 2 * len(DEV_SEEDS) * DEV_ROUNDS + 4 * len(TEST_SEEDS) * TEST_ROUNDS)

## 4. Safety shield and features

The safety shield filters actions before the learned policy chooses among them. It checks whether an action leaves a viable future path instead of selecting an action by itself.

V4 includes:

- framework-compatible blast geometry and countdown timing;
- short-horizon opponent reachability;
- corridor depth and progress toward junctions;
- bomb placement checks that require a usable escape route;
- persistent escape-path features that are revalidated each step;
- a fallback action based on predicted survival time.


In [ ]:
%%writefile /content/bomberman_rl/agent_code/survival_linear_ppo_v4/core.py
"""Safety mask and feature extraction for the V4 agent."""
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import math

import numpy as np

ACTIONS = ["UP", "RIGHT", "DOWN", "LEFT", "WAIT", "BOMB"]
MOVE_ACTIONS = ACTIONS[:4]
DELTAS = {
    "UP": (0, -1), "RIGHT": (1, 0), "DOWN": (0, 1),
    "LEFT": (-1, 0), "WAIT": (0, 0), "BOMB": (0, 0),
}
REVERSE = {"UP": "DOWN", "DOWN": "UP", "LEFT": "RIGHT", "RIGHT": "LEFT"}
HORIZON = 8

FEATURE_NAMES = [
    "survival_breadth", "terminal_width", "blast_margin", "survival_duration",
    "danger_improvement", "coin_progress", "crate_progress", "opponent_progress",
    "free_neighbors", "corridor_exit", "dead_end_risk", "opponent_pressure",
    "contested_now", "contested_path", "visit_penalty", "reverse_action",
    "plan_match", "own_blast_exit", "move", "safe_wait", "bomb",
    "bomb_crates", "bomb_opponents", "bomb_has_target", "bomb_escape_speed",
    "bomb_escape_breadth", "future_bomb_value", "trap_opportunity",
]

STATE_FEATURE_NAMES = [
    "bias", "score", "bomb_available", "coins", "crates", "opponents",
    "danger", "coin_distance", "crate_distance", "opponent_distance",
    "free_neighbors", "corridor_depth", "step_fraction", "legal_fraction",
    "best_coin_progress", "best_crate_progress", "best_opponent_progress",
    "best_bomb_value", "escape_mode", "recent_unique_fraction",
]


@dataclass(frozen=True)
class EscapeCertificate:
    safe: bool
    duration: int
    breadth: float
    terminal_width: int
    min_margin: float
    contested_fraction: float
    exit_step: int
    path: tuple


def inside(field, pos):
    x, y = pos
    return 0 <= x < field.shape[0] and 0 <= y < field.shape[1]


def neighbors(pos):
    x, y = pos
    return ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1))


def blast_tiles(field, origin, power=3):
    """Match framework blast geometry; stone walls stop a blast."""
    tiles = [tuple(origin)]
    ox, oy = origin
    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        for distance in range(1, power + 1):
            pos = (ox + dx * distance, oy + dy * distance)
            if not inside(field, pos) or field[pos] == -1:
                break
            tiles.append(pos)
    return tiles


def _bomb_data(game_state, extra_bomb=None):
    bombs = [(tuple(pos), int(timer)) for pos, timer in game_state.get("bombs", [])]
    if extra_bomb is not None:
        bombs.append((tuple(extra_bomb), 4))
    detonations = {}
    for pos, timer in bombs:
        detonations[pos] = min(detonations.get(pos, 99), max(1, timer + 1))
    return bombs, detonations


def danger_schedule(game_state, extra_bomb=None, horizon=HORIZON):
    field = game_state["field"]
    bombs, detonations = _bomb_data(game_state, extra_bomb)
    horizon = max(horizon, max(detonations.values(), default=1) + 1)
    danger = np.zeros((horizon + 1, *field.shape), dtype=bool)
    explosion = np.asarray(game_state.get("explosion_map", np.zeros_like(field)))
    danger[1][explosion > 0] = True
    blast_lookup = {}
    for pos, _ in bombs:
        blast = tuple(blast_tiles(field, pos))
        blast_lookup[pos] = blast
        detonation = detonations[pos]
        for time_index in (detonation, detonation + 1):
            if time_index <= horizon:
                for tile in blast:
                    danger[(time_index, *tile)] = True
    return danger, detonations, blast_lookup


def earliest_blast_time(game_state, pos):
    field = game_state["field"]
    if np.asarray(game_state.get("explosion_map", np.zeros_like(field)))[tuple(pos)] > 0:
        return 0.0
    earliest = 9.0
    for bomb_pos, timer in game_state.get("bombs", []):
        if tuple(pos) in blast_tiles(field, tuple(bomb_pos)):
            earliest = min(earliest, float(timer))
    return earliest


def static_free_tiles(game_state):
    field = game_state["field"]
    free = field == 0
    return free


def opponent_reachability(game_state, horizon=HORIZON, extra_bomb=None):
    """Estimate tiles an opponent can reach at each future step."""
    field = game_state["field"]
    _, detonations = _bomb_data(game_state, extra_bomb)
    current = {tuple(other[3]) for other in game_state.get("others", [])}
    reachable = [set(current)]
    for time_index in range(1, horizon + 1):
        nxt_set = set()
        for pos in current:
            for nxt in (pos, *neighbors(pos)):
                if not inside(field, nxt) or field[nxt] != 0:
                    continue
                if nxt != pos and nxt in detonations and time_index < detonations[nxt]:
                    continue
                nxt_set.add(nxt)
        current = nxt_set
        reachable.append(set(current))
    return reachable


def _free_degree(field, pos):
    return sum(inside(field, nxt) and field[nxt] == 0 for nxt in neighbors(pos))


def corridor_depth(field, start, max_depth=6):
    """Distance to the nearest junction; larger values mean deeper corridors."""
    start = tuple(start)
    if not inside(field, start) or field[start] != 0:
        return float(max_depth)
    if _free_degree(field, start) >= 3:
        return 0.0
    queue = deque([(start, 0)])
    seen = {start}
    while queue:
        pos, depth = queue.popleft()
        if depth >= max_depth:
            continue
        for nxt in neighbors(pos):
            if not inside(field, nxt) or field[nxt] != 0 or nxt in seen:
                continue
            if _free_degree(field, nxt) >= 3:
                return float(depth + 1)
            seen.add(nxt)
            queue.append((nxt, depth + 1))
    return float(max_depth)


def target_approaches(game_state, targets, targets_are_blocked=False):
    field = game_state["field"]
    result = set()
    for target in map(tuple, targets):
        if inside(field, target) and field[target] == 0 and not targets_are_blocked:
            result.add(target)
        else:
            result.update(nxt for nxt in neighbors(target) if inside(field, nxt) and field[nxt] == 0)
    return result


def shortest_distance(game_state, start, targets, max_depth=40):
    targets = set(map(tuple, targets))
    start = tuple(start)
    if not targets:
        return float(max_depth)
    if start in targets:
        return 0.0
    field = game_state["field"]
    blocked = {tuple(pos) for pos, _ in game_state.get("bombs", [])}
    blocked.update(tuple(other[3]) for other in game_state.get("others", []))
    queue = deque([(start, 0)])
    seen = {start}
    while queue:
        pos, distance = queue.popleft()
        if distance >= max_depth:
            continue
        for nxt in neighbors(pos):
            if nxt in seen or not inside(field, nxt) or field[nxt] != 0:
                continue
            if nxt in targets:
                return float(distance + 1)
            if nxt in blocked:
                continue
            seen.add(nxt)
            queue.append((nxt, distance + 1))
    return float(max_depth)


def escape_certificate(game_state, start, extra_bomb=None, horizon=HORIZON, risk_averse=False):
    """Search future states for a survivable path."""
    field = game_state["field"]
    start = tuple(start)
    danger, detonations, blast_lookup = danger_schedule(game_state, extra_bomb, horizon)
    horizon = danger.shape[0] - 1
    opponent_tiles = opponent_reachability(game_state, horizon, extra_bomb)
    new_blast = set(blast_lookup.get(tuple(extra_bomb), ())) if extra_bomb is not None else set()

    if not inside(field, start) or field[start] != 0 or danger[(1, *start)]:
        return EscapeCertificate(False, 0, 0.0, 0, 0.0, 1.0, 99, ())

    initial_contested = start in opponent_tiles[1]
    if risk_averse and initial_contested:
        return EscapeCertificate(False, 0, 0.0, 0, 0.0, 1.0, 99, (start,))

    # position -> (path, opponent conflicts, minimum blast margin)
    frontier = {start: ((start,), int(initial_contested), 9.0)}
    widths = [1]
    duration = 1
    for time_index in range(1, horizon):
        candidates = {}
        for pos, (path, contested, min_margin) in frontier.items():
            for nxt in (pos, *neighbors(pos)):
                if not inside(field, nxt) or field[nxt] != 0:
                    continue
                # The current bomb tile may be occupied once, but cannot be re-entered.
                if nxt != pos and nxt in detonations and time_index + 1 < detonations[nxt]:
                    continue
                if danger[(time_index + 1, *nxt)]:
                    continue
                is_contested = nxt in opponent_tiles[min(time_index + 1, len(opponent_tiles) - 1)]
                # Treat near-term opponent collisions as blocked while escaping.
                if risk_averse and time_index + 1 <= 2 and is_contested:
                    continue
                margins = []
                for bomb_pos, detonation in detonations.items():
                    if nxt in blast_lookup[bomb_pos] and detonation >= time_index + 1:
                        margins.append(detonation - (time_index + 1))
                local_margin = float(min(margins, default=9))
                candidate = (
                    path + (nxt,),
                    contested + int(is_contested),
                    min(min_margin, local_margin),
                )
                old = candidates.get(nxt)
                candidate_exit = next((i for i, tile in enumerate(candidate[0], start=1) if tile not in new_blast), 99) if new_blast else 0
                old_exit = next((i for i, tile in enumerate(old[0], start=1) if tile not in new_blast), 99) if old and new_blast else 0
                rank = (candidate_exit, candidate[1], -candidate[2], corridor_depth(field, nxt))
                old_rank = (old_exit, old[1], -old[2], corridor_depth(field, nxt)) if old else None
                if old is None or rank < old_rank:
                    candidates[nxt] = candidate
        frontier = candidates
        if not frontier:
            break
        duration = time_index + 1
        widths.append(len(frontier))

    if frontier:
        best_pos, best_data = min(
            frontier.items(),
            key=lambda item: (
                next((i for i, tile in enumerate(item[1][0], start=1) if tile not in new_blast), 99) if new_blast else 0,
                item[1][1], corridor_depth(field, item[0]), -item[1][2], -_free_degree(field, item[0]),
            ),
        )
        path, contested, min_margin = best_data
    else:
        path, contested, min_margin = (), horizon, 0.0

    exit_step = 0
    if new_blast and path:
        exit_step = next((index for index, pos in enumerate(path, start=1) if pos not in new_blast), 99)
    breadth = float(np.mean([min(1.0, width / 8.0) for width in widths]))
    safe = duration >= horizon
    return EscapeCertificate(
        safe=safe,
        duration=duration,
        breadth=breadth,
        terminal_width=len(frontier),
        min_margin=float(min_margin),
        contested_fraction=float(contested / max(1, len(path))),
        exit_step=int(exit_step),
        path=tuple(path),
    )


def bomb_targets(game_state, origin):
    field = game_state["field"]
    blast = set(blast_tiles(field, tuple(origin)))
    crates = {tuple(pos) for pos in np.argwhere(field == 1)}
    opponents = {tuple(other[3]) for other in game_state.get("others", [])}
    return sum(tile in crates for tile in blast), sum(tile in opponents for tile in blast)


def _physical_actions(game_state):
    field = game_state["field"]
    _, _, bomb_available, pos = game_state["self"]
    pos = tuple(pos)
    occupied = {tuple(p) for p, _ in game_state.get("bombs", [])}
    occupied.update(tuple(other[3]) for other in game_state.get("others", []))
    physical = np.zeros(len(ACTIONS), dtype=bool)
    for index, action in enumerate(ACTIONS):
        dx, dy = DELTAS[action]
        nxt = (pos[0] + dx, pos[1] + dy)
        if action in MOVE_ACTIONS:
            physical[index] = inside(field, nxt) and field[nxt] == 0 and nxt not in occupied
        elif action == "WAIT":
            physical[index] = True
        else:
            physical[index] = bool(bomb_available) and pos not in occupied
    return physical


def safe_action_data(game_state, context=None):
    """Return action features, mask values and per-action safety details."""
    context = context or {}
    shield_mode = context.get("shield_mode", "full")
    if shield_mode not in {"full", "basic"}:
        raise ValueError(f"Unknown shield mode: {shield_mode}")
    field = game_state["field"]
    _, _, bomb_available, pos = game_state["self"]
    pos = tuple(pos)
    coins = [tuple(item) for item in game_state.get("coins", [])]
    crates = [tuple(item) for item in np.argwhere(field == 1)]
    opponents = [tuple(other[3]) for other in game_state.get("others", [])]
    coin_targets = target_approaches(game_state, coins)
    crate_targets = target_approaches(game_state, crates, targets_are_blocked=True)
    opponent_targets = target_approaches(game_state, opponents, targets_are_blocked=True)
    current_distances = (
        shortest_distance(game_state, pos, coin_targets),
        shortest_distance(game_state, pos, crate_targets),
        shortest_distance(game_state, pos, opponent_targets),
    )
    current_danger = earliest_blast_time(game_state, pos)
    current_corridor = corridor_depth(field, pos)
    own_bombs = {tuple(item) for item in context.get("own_bombs", set())}
    own_blast = set()
    for own_bomb in own_bombs:
        own_blast.update(blast_tiles(field, own_bomb))
    escape_mode = bool(own_bombs) or current_danger <= 3
    recent = [tuple(item) for item in context.get("recent_positions", ())]
    previous_action = context.get("previous_action")
    plan = [tuple(item) for item in context.get("escape_plan", ())]
    planned_next = plan[0] if plan else None
    physical = _physical_actions(game_state)
    phi = np.zeros((len(ACTIONS), len(FEATURE_NAMES)), dtype=np.float64)
    legal = np.zeros(len(ACTIONS), dtype=bool)
    details = []

    for index, action in enumerate(ACTIONS):
        dx, dy = DELTAS[action]
        nxt = (pos[0] + dx, pos[1] + dy)
        extra_bomb = pos if action == "BOMB" and physical[index] else None
        cert = escape_certificate(
            game_state, nxt, extra_bomb=extra_bomb,
            risk_averse=escape_mode or action == "BOMB",
        ) if physical[index] else EscapeCertificate(False, 0, 0.0, 0, 0.0, 1.0, 99, ())
        crate_hits, opponent_hits = bomb_targets(game_state, pos) if action == "BOMB" else (0, 0)
        has_target = crate_hits + opponent_hits > 0
        robust_bomb = (
            action != "BOMB" or (
                bool(bomb_available) and has_target and cert.safe
                and 1 <= cert.exit_step <= 4
                and cert.terminal_width >= 2
                and cert.breadth >= 0.20
            )
        )
        legal[index] = bool(physical[index] and cert.safe and robust_bomb)

        next_distances = (
            shortest_distance(game_state, nxt, coin_targets) if physical[index] else current_distances[0],
            shortest_distance(game_state, nxt, crate_targets) if physical[index] else current_distances[1],
            shortest_distance(game_state, nxt, opponent_targets) if physical[index] else current_distances[2],
        )
        progress = [np.tanh((old - new) / 3.0) for old, new in zip(current_distances, next_distances)]
        next_danger = earliest_blast_time(game_state, nxt) if physical[index] else 0.0
        next_corridor = corridor_depth(field, nxt) if physical[index] else 6.0
        nearest_opponent = min((abs(nxt[0] - ox) + abs(nxt[1] - oy) for ox, oy in opponents), default=20)
        pressure = math.exp(-nearest_opponent / 3.0)
        free_neighbors = sum(
            inside(field, candidate) and field[candidate] == 0 for candidate in neighbors(nxt)
        ) / 4.0 if inside(field, nxt) else 0.0
        future_crates = future_opponents = 0
        if physical[index] and action in MOVE_ACTIONS and bomb_available:
            future_crates, future_opponents = bomb_targets(game_state, nxt)
        future_bomb_value = min(1.0, (future_crates + 3 * future_opponents) / 5.0)
        trap_opportunity = float(opponent_hits > 0) * min(1.0, next_corridor / 4.0)
        current_in_own_blast = pos in own_blast
        next_in_own_blast = nxt in own_blast

        phi[index] = [
            cert.breadth,
            min(1.0, cert.terminal_width / 8.0),
            min(1.0, cert.min_margin / 5.0),
            min(1.0, cert.duration / HORIZON),
            np.clip((next_danger - current_danger) / 4.0, -1.0, 1.0),
            progress[0], progress[1], progress[2], free_neighbors,
            np.clip((current_corridor - next_corridor) / 6.0, -1.0, 1.0),
            min(1.0, next_corridor / 6.0) * pressure,
            pressure,
            float(cert.contested_fraction > 0 and len(cert.path) > 0 and cert.path[0] in opponent_reachability(game_state, 1)[1]),
            cert.contested_fraction,
            min(1.0, recent[-16:].count(nxt) / 4.0),
            float(action == REVERSE.get(previous_action)),
            float(planned_next is not None and nxt == planned_next),
            float(current_in_own_blast) - float(next_in_own_blast),
            float(action in MOVE_ACTIONS),
            float(action == "WAIT" and cert.safe),
            float(action == "BOMB"),
            min(1.0, crate_hits / 4.0),
            min(1.0, opponent_hits / 2.0),
            float(has_target),
            0.0 if action != "BOMB" or cert.exit_step >= 99 else (5.0 - cert.exit_step) / 4.0,
            cert.breadth if action == "BOMB" else 0.0,
            future_bomb_value,
            trap_opportunity,
        ]
        details.append(cert)

    if shield_mode == "basic":
        # Basic mode keeps the same features but uses fewer mask checks.
        immediate, _, _ = danger_schedule(game_state)
        basic = physical.copy()
        for i, action in enumerate(ACTIONS):
            dx, dy = DELTAS[action]
            nxt = (pos[0] + dx, pos[1] + dy)
            if physical[i]:
                basic[i] = not immediate[(1, *nxt)]
        fallback = not basic.any()
        if fallback:
            basic = physical.copy()  # No certificate-based least-risk ranking.
        return phi, basic, {"certificates": details, "fallback": fallback,
                            "escape_mode": escape_mode, "shield_mode": "basic"}

    # Prefer leaving the blast area when a safe immediate exit is available.
    if own_bombs and legal.any() and pos in own_blast:
        exits = legal & (phi[:, FEATURE_NAMES.index("own_blast_exit")] > 0)
        if exits.any():
            legal = exits

    fallback = False
    if not legal.any():
        # Rank fallback actions by survival time, opponent conflict and route width.
        fallback = True
        candidates = np.flatnonzero(physical)
        if len(candidates) == 0:
            candidates = np.array([ACTIONS.index("WAIT")])
        ranks = [
            (details[i].duration, -details[i].contested_fraction, details[i].terminal_width, details[i].breadth)
            for i in candidates
        ]
        best_rank = max(ranks)
        for i, rank in zip(candidates, ranks):
            if rank == best_rank:
                legal[i] = True

    return phi, legal, {"certificates": details, "fallback": fallback, "escape_mode": escape_mode}


def state_features(game_state, context=None, action_data=None):
    if game_state is None:
        return np.zeros(len(STATE_FEATURE_NAMES), dtype=np.float64)
    context = context or {}
    field = game_state["field"]
    _, score, bomb_available, pos = game_state["self"]
    pos = tuple(pos)
    coins = [tuple(item) for item in game_state.get("coins", [])]
    crates = [tuple(item) for item in np.argwhere(field == 1)]
    opponents = [tuple(other[3]) for other in game_state.get("others", [])]
    coin_targets = target_approaches(game_state, coins)
    crate_targets = target_approaches(game_state, crates, True)
    opponent_targets = target_approaches(game_state, opponents, True)
    if action_data is None:
        phi, legal, metadata = safe_action_data(game_state, context)
    else:
        phi, legal, metadata = action_data
    recent = [tuple(item) for item in context.get("recent_positions", ())]
    unique_fraction = len(set(recent[-16:])) / max(1, len(recent[-16:]))
    return np.asarray([
        1.0,
        np.tanh(float(score) / 10.0),
        float(bool(bomb_available)),
        min(1.0, len(coins) / 20.0),
        min(1.0, len(crates) / 100.0),
        len(opponents) / 3.0,
        np.clip((4.0 - earliest_blast_time(game_state, pos)) / 4.0, 0.0, 1.0),
        np.tanh(shortest_distance(game_state, pos, coin_targets) / 10.0),
        np.tanh(shortest_distance(game_state, pos, crate_targets) / 10.0),
        np.tanh(shortest_distance(game_state, pos, opponent_targets) / 10.0),
        _free_degree(field, pos) / 4.0,
        corridor_depth(field, pos) / 6.0,
        min(1.0, game_state.get("step", 0) / 400.0),
        float(legal.mean()),
        float(phi[legal, FEATURE_NAMES.index("coin_progress")].max(initial=0.0)),
        float(phi[legal, FEATURE_NAMES.index("crate_progress")].max(initial=0.0)),
        float(phi[legal, FEATURE_NAMES.index("opponent_progress")].max(initial=0.0)),
        float((phi[legal, FEATURE_NAMES.index("bomb_crates")] + 3 * phi[legal, FEATURE_NAMES.index("bomb_opponents")]).max(initial=0.0)),
        float(metadata["escape_mode"]),
        unique_fraction,
    ], dtype=np.float64)


def action_matrix(game_state, context=None):
    action_data = safe_action_data(game_state, context)
    phi, legal, metadata = action_data
    state = state_features(game_state, context, action_data)
    matrix = np.concatenate([phi, np.repeat(state[None, :], len(ACTIONS), axis=0)], axis=1)
    return matrix, legal, metadata

## 5. Linear PPO model

The actor is linear: action logits are `phi(s,a) @ theta`, and the critic uses `v(s) @ w`. Directional actions share action-relative features. Optimizer state is saved with the checkpoint so training can continue across curriculum stages.


In [ ]:
%%writefile /content/bomberman_rl/agent_code/survival_linear_ppo_v4/model.py
"""Linear PPO actor-critic with saved optimizer state."""
from __future__ import annotations

import os
from pathlib import Path
import pickle

import numpy as np

from .core import ACTIONS, FEATURE_NAMES, STATE_FEATURE_NAMES, safe_action_data, state_features

DEFAULT_CONFIG = {
    "gamma": 0.99, "gae_lambda": 0.95, "clip_epsilon": 0.20,
    "actor_lr": 4e-3, "critic_lr": 3e-3, "entropy_coef": 0.010,
    "ppo_epochs": 4, "minibatch_size": 256, "min_update_steps": 256,
    "target_kl": 0.012, "max_grad_norm": 0.75, "seed": 7301,
}


def softmax_masked(logits, mask):
    logits = np.where(mask, logits, -1e9)
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exponentials = np.exp(np.clip(logits, -60, 60)) * mask
    return exponentials / np.maximum(exponentials.sum(axis=-1, keepdims=True), 1e-12)


class Adam:
    def __init__(self, shape, lr):
        self.lr = float(lr)
        self.initial_lr = float(lr)
        self.m = np.zeros(shape, dtype=np.float64)
        self.v = np.zeros(shape, dtype=np.float64)
        self.t = 0

    def step(self, parameter, gradient, max_norm):
        norm = float(np.linalg.norm(gradient))
        if norm > max_norm:
            gradient = gradient * max_norm / (norm + 1e-12)
        self.t += 1
        self.m = 0.9 * self.m + 0.1 * gradient
        self.v = 0.999 * self.v + 0.001 * gradient * gradient
        m_hat = self.m / (1 - 0.9 ** self.t)
        v_hat = self.v / (1 - 0.999 ** self.t)
        parameter += self.lr * m_hat / (np.sqrt(v_hat) + 1e-8)

    def state_dict(self):
        return {"lr": self.lr, "initial_lr": self.initial_lr, "m": self.m, "v": self.v, "t": self.t}

    def load_state_dict(self, state):
        if not state:
            return
        self.lr = float(state["lr"])
        self.initial_lr = float(state.get("initial_lr", self.initial_lr))
        self.m[:] = state["m"]
        self.v[:] = state["v"]
        self.t = int(state["t"])


class LinearPPO:
    def __init__(self, config=None):
        self.config = {**DEFAULT_CONFIG, **(config or {})}
        self.theta = np.zeros(len(FEATURE_NAMES), dtype=np.float64)
        self.value_w = np.zeros(len(STATE_FEATURE_NAMES), dtype=np.float64)
        priors = {
            "survival_breadth": 1.2, "terminal_width": 0.5, "blast_margin": 0.5,
            "survival_duration": 1.0, "danger_improvement": 0.8,
            "coin_progress": 0.35, "crate_progress": 0.25, "opponent_progress": 0.10,
            "free_neighbors": 0.20, "corridor_exit": 0.35, "dead_end_risk": -0.8,
            "opponent_pressure": -0.10, "contested_now": -1.0, "contested_path": -0.5,
            "visit_penalty": -0.25, "reverse_action": -0.20, "plan_match": 0.75,
            "own_blast_exit": 1.25, "move": 0.10, "safe_wait": -0.05, "bomb": -0.30,
            "bomb_crates": 0.60, "bomb_opponents": 1.20, "bomb_has_target": 0.25,
            "bomb_escape_speed": 0.60, "bomb_escape_breadth": 0.75,
            "future_bomb_value": 0.25, "trap_opportunity": 0.50,
        }
        for name, value in priors.items():
            self.theta[FEATURE_NAMES.index(name)] = value
        self.n_updates = 0
        self.actor_opt = Adam(self.theta.shape, self.config["actor_lr"])
        self.critic_opt = Adam(self.value_w.shape, self.config["critic_lr"])

    def probs(self, phi, mask):
        return softmax_masked(np.einsum("...ad,d->...a", phi, self.theta), mask)

    def value(self, state_vector):
        return np.asarray(state_vector) @ self.value_w

    def choose(self, game_state, rng, stochastic=True, context=None):
        context = {**(context or {}), "shield_mode": self.config.get("shield_mode", "full")}
        phi, mask, metadata = safe_action_data(game_state, context)
        state_vector = state_features(game_state, context, (phi, mask, metadata))
        probabilities = self.probs(phi, mask)
        action_index = int(rng.choice(len(ACTIONS), p=probabilities)) if stochastic else int(np.argmax(probabilities))
        record = {
            "phi": phi, "mask": mask, "state": state_vector,
            "action": action_index, "old_logp": float(np.log(probabilities[action_index] + 1e-12)),
            "old_probs": probabilities.copy(), "value": float(self.value(state_vector)),
            "reward": None, "next_value": 0.0, "done": False,
        }
        decision = {
            "certificate": metadata["certificates"][action_index],
            "fallback": metadata["fallback"],
        }
        return ACTIONS[action_index], record, decision

    def _gae(self, records):
        rewards = np.asarray([item["reward"] for item in records], dtype=float)
        values = np.asarray([item["value"] for item in records], dtype=float)
        next_values = np.asarray([item["next_value"] for item in records], dtype=float)
        dones = np.asarray([item["done"] for item in records], dtype=float)
        deltas = rewards + self.config["gamma"] * next_values * (1 - dones) - values
        advantages = np.zeros_like(deltas)
        running = 0.0
        for index in range(len(records) - 1, -1, -1):
            running = deltas[index] + self.config["gamma"] * self.config["gae_lambda"] * (1 - dones[index]) * running
            advantages[index] = running
        returns = np.clip(advantages + values, -20.0, 20.0)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return advantages, returns

    def _exact_kl(self, phi, mask, old_probs):
        new_probs = self.probs(phi, mask)
        return float(np.mean(np.sum(old_probs * (np.log(old_probs + 1e-12) - np.log(new_probs + 1e-12)), axis=1)))

    def update(self, records, rng):
        records = [item for item in records if item["reward"] is not None]
        if not records:
            return {}
        phi = np.stack([item["phi"] for item in records])
        mask = np.stack([item["mask"] for item in records])
        state = np.stack([item["state"] for item in records])
        actions = np.asarray([item["action"] for item in records], dtype=int)
        old_logp = np.asarray([item["old_logp"] for item in records])
        old_probs = np.stack([item["old_probs"] for item in records])
        advantages, returns = self._gae(records)
        n_items = len(records)
        last_kl = 0.0
        clip_fractions = []

        for _ in range(self.config["ppo_epochs"]):
            epoch_start = self.theta.copy()
            shuffled = rng.permutation(n_items)
            for start in range(0, n_items, self.config["minibatch_size"]):
                indices = shuffled[start:start + self.config["minibatch_size"]]
                probabilities = self.probs(phi[indices], mask[indices])
                chosen = probabilities[np.arange(len(indices)), actions[indices]]
                ratio = np.exp(np.log(chosen + 1e-12) - old_logp[indices])
                advantage = advantages[indices]
                clipped = ((advantage >= 0) & (ratio > 1 + self.config["clip_epsilon"])) | ((advantage < 0) & (ratio < 1 - self.config["clip_epsilon"]))
                clip_fractions.append(float(clipped.mean()))
                expected_phi = np.einsum("ba,bad->bd", probabilities, phi[indices])
                score_gradient = phi[indices, actions[indices]] - expected_phi
                actor_gradient = np.mean((~clipped)[:, None] * ratio[:, None] * advantage[:, None] * score_gradient, axis=0)
                entropy = -np.sum(probabilities * np.log(probabilities + 1e-12), axis=1)
                entropy_logit_gradient = -probabilities * (np.log(probabilities + 1e-12) + entropy[:, None])
                actor_gradient += self.config["entropy_coef"] * np.mean(
                    np.einsum("ba,bad->bd", entropy_logit_gradient, phi[indices]), axis=0
                )
                self.actor_opt.step(self.theta, actor_gradient, self.config["max_grad_norm"])

                prediction = state[indices] @ self.value_w
                critic_error = np.clip(returns[indices] - prediction, -1.0, 1.0)  # Huber gradient
                critic_gradient = np.mean(critic_error[:, None] * state[indices], axis=0)
                self.critic_opt.step(self.value_w, critic_gradient, self.config["max_grad_norm"])

            last_kl = self._exact_kl(phi, mask, old_probs)
            if last_kl > 1.5 * self.config["target_kl"]:
                candidate = self.theta.copy()
                for fraction in (0.5, 0.25, 0.125, 0.0):
                    self.theta[:] = epoch_start + fraction * (candidate - epoch_start)
                    if self._exact_kl(phi, mask, old_probs) <= self.config["target_kl"]:
                        break
                last_kl = self._exact_kl(phi, mask, old_probs)
                break

        if last_kl > 1.25 * self.config["target_kl"]:
            self.actor_opt.lr = max(self.actor_opt.initial_lr * 0.10, self.actor_opt.lr * 0.70)
        elif last_kl < 0.50 * self.config["target_kl"]:
            self.actor_opt.lr = min(self.actor_opt.initial_lr, self.actor_opt.lr * 1.03)
        self.n_updates += 1
        final_probs = self.probs(phi, mask)
        final_entropy = float(np.mean(-np.sum(final_probs * np.log(final_probs + 1e-12), axis=1)))
        return {
            "update": self.n_updates, "steps": n_items,
            "reward_mean": float(np.mean([item["reward"] for item in records])),
            "return_mean": float(returns.mean()),
            "value_mse": float(np.mean((state @ self.value_w - returns) ** 2)),
            "entropy": final_entropy, "kl": last_kl,
            "clip_fraction": float(np.mean(clip_fractions)) if clip_fractions else 0.0,
            "actor_lr": self.actor_opt.lr,
        }

    def save(self, path):
        path = Path(path)
        payload = {
            "theta": self.theta, "value_w": self.value_w, "config": self.config,
            "n_updates": self.n_updates, "feature_names": FEATURE_NAMES,
            "state_feature_names": STATE_FEATURE_NAMES,
            "actor_opt": self.actor_opt.state_dict(), "critic_opt": self.critic_opt.state_dict(),
            "format_version": 4,
        }
        temporary = path.with_suffix(path.suffix + ".tmp")
        with temporary.open("wb") as stream:
            pickle.dump(payload, stream)
        os.replace(temporary, path)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as stream:
            payload = pickle.load(stream)
        if payload.get("feature_names") != FEATURE_NAMES or payload.get("state_feature_names") != STATE_FEATURE_NAMES:
            raise ValueError("Saved V4 feature schema does not match this code")
        model = cls(payload.get("config"))
        model.theta[:] = payload["theta"]
        model.value_w[:] = payload["value_w"]
        model.n_updates = int(payload.get("n_updates", 0))
        model.actor_opt.load_state_dict(payload.get("actor_opt"))
        model.critic_opt.load_state_dict(payload.get("critic_opt"))
        return model

In [ ]:
%%writefile /content/bomberman_rl/agent_code/survival_linear_ppo_v4/callbacks.py
"""Callbacks for the shielded linear PPO V4 agent."""
from collections import deque
from pathlib import Path

import numpy as np

from .model import LinearPPO

MODEL_PATH = Path(__file__).with_name("model.pkl")


def setup(self):
    self.rng = np.random.default_rng(7301)
    self.own_bomb_positions = set()
    self.recent_positions = deque(maxlen=20)
    self.escape_plan = []
    self.expected_position = None
    self.previous_action = None
    self.current_round = None
    self.shield_fallbacks = 0
    import os
    checkpoint = MODEL_PATH if self.train else Path(os.environ.get("BOMBERMAN_EVAL_MODEL_PATH", str(MODEL_PATH)))
    if not self.train and not checkpoint.exists():
        raise FileNotFoundError(f"Evaluation requires an explicit checkpoint: {checkpoint}")
    self.model = LinearPPO.load(checkpoint) if checkpoint.exists() else LinearPPO()
    self.logger.info("Linear PPO V4 ready at update %d", self.model.n_updates)


def _context(self):
    return {
        "own_bombs": set(self.own_bomb_positions),
        "recent_positions": tuple(self.recent_positions),
        "escape_plan": tuple(self.escape_plan),
        "previous_action": self.previous_action,
    }


def act(self, game_state: dict) -> str:
    position = tuple(game_state["self"][3])
    if self.current_round != game_state["round"]:
        self.current_round = game_state["round"]
        self.own_bomb_positions.clear()
        self.recent_positions.clear()
        self.escape_plan.clear()
        self.expected_position = None
        self.previous_action = None
    if self.expected_position is not None and position != self.expected_position:
        self.escape_plan.clear()  # previous move was blocked or the plan became stale
    live_bombs = {tuple(pos) for pos, _ in game_state.get("bombs", [])}
    self.own_bomb_positions.intersection_update(live_bombs)
    action, record, decision = self.model.choose(
        game_state, self.rng, stochastic=bool(self.train), context=_context(self)
    )
    certificate = decision["certificate"]
    self.escape_plan = list(certificate.path[1:]) if certificate.path else []
    self.expected_position = certificate.path[0] if certificate.path else position
    if decision["fallback"]:
        self.shield_fallbacks += 1
    if self.train:
        self.rollout.append(record)
    if action == "BOMB":
        self.own_bomb_positions.add(position)
        if self.train:
            self.pending_bombs.append({"position": position, "record": record})
    self.recent_positions.append(position)
    self.previous_action = action
    return action

In [ ]:
%%writefile /content/bomberman_rl/agent_code/survival_linear_ppo_v4/train.py
"""Training callbacks for the linear PPO V4 agent."""
from pathlib import Path
import json

import numpy as np
import events as e

from .core import ACTIONS, corridor_depth, earliest_blast_time, shortest_distance, state_features, target_approaches

MODEL_PATH = Path(__file__).with_name("model.pkl")
METRICS_PATH = Path(__file__).with_name("training_metrics.jsonl")

EVENT_REWARDS = {
    e.COIN_COLLECTED: 1.0,
    e.KILLED_OPPONENT: 5.0,
    e.SURVIVED_ROUND: 2.0,
    e.INVALID_ACTION: -1.0,
    e.GOT_KILLED: -4.0,
    e.KILLED_SELF: -8.0,
}


def setup_training(self):
    self.rollout = []
    self.pending_bombs = []


def _context(self):
    return {
        "own_bombs": set(self.own_bomb_positions),
        "recent_positions": tuple(self.recent_positions),
        "escape_plan": tuple(self.escape_plan),
        "previous_action": self.previous_action,
    }


def _potential(game_state):
    if game_state is None:
        return 0.0
    field = game_state["field"]
    pos = tuple(game_state["self"][3])
    coins = target_approaches(game_state, game_state.get("coins", []))
    crates = target_approaches(game_state, np.argwhere(field == 1), True)
    opponents = target_approaches(game_state, [other[3] for other in game_state.get("others", [])], True)
    closeness = lambda distance: 1.0 - min(40.0, distance) / 40.0
    target = (
        0.55 * closeness(shortest_distance(game_state, pos, coins))
        + 0.30 * closeness(shortest_distance(game_state, pos, crates))
        + 0.15 * closeness(shortest_distance(game_state, pos, opponents))
    )
    safety = 0.25 * float(earliest_blast_time(game_state, pos) > 1)
    corridor = -0.10 * corridor_depth(field, pos) / 6.0
    return float(target + safety + corridor)


def _transition_reward(self, old_state, new_state, events):
    sparse = sum(EVENT_REWARDS.get(event, 0.0) for event in events)
    shaped = 0.99 * _potential(new_state) - _potential(old_state)
    repeat_penalty = 0.0
    if new_state is not None:
        new_pos = tuple(new_state["self"][3])
        repeat_penalty = -0.03 * min(4, tuple(self.recent_positions).count(new_pos))
    return float(sparse + 0.25 * shaped + repeat_penalty)


def _resolve_bomb(self, events, terminal=False):
    if not self.pending_bombs:
        return
    exploded = e.BOMB_EXPLODED in events
    self_killed = e.KILLED_SELF in events
    if not (exploded or self_killed or terminal):
        return
    pending = self.pending_bombs.pop(0)
    if self_killed:
        bonus = -8.0
    elif exploded:
        crates = events.count(e.CRATE_DESTROYED)
        kills = events.count(e.KILLED_OPPONENT)
        found = events.count(e.COIN_FOUND)
        bonus = 5.0 * kills + 0.25 * crates + 0.10 * found
        bonus += 0.20 if crates + kills else -0.25
    else:
        bonus = 0.0
    record = pending["record"]
    if record["reward"] is None:
        record["delayed_bonus"] = record.get("delayed_bonus", 0.0) + bonus
    else:
        record["reward"] += bonus


def _finish_latest(self, old_state, new_state, events, done):
    if not self.rollout or self.rollout[-1]["reward"] is not None:
        return
    record = self.rollout[-1]
    record["reward"] = _transition_reward(self, old_state, new_state, events) + record.pop("delayed_bonus", 0.0)
    record["done"] = bool(done)
    record["next_value"] = 0.0 if done or new_state is None else float(self.model.value(state_features(new_state, _context(self))))


def _maybe_update(self, force=False):
    if self.pending_bombs:
        return
    minimum = int(self.model.config["min_update_steps"])
    if len(self.rollout) < minimum and not force:
        return
    if not self.rollout:
        return
    metrics = self.model.update(self.rollout, self.rng)
    if metrics:
        with METRICS_PATH.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(metrics) + "\n")
        self.logger.info("PPO V4 update=%d steps=%d KL=%.5f", metrics["update"], metrics["steps"], metrics["kl"])
    self.rollout.clear()


def game_events_occurred(self, old_game_state, self_action, new_game_state, events):
    _resolve_bomb(self, events)
    _finish_latest(self, old_game_state, new_game_state, events, done=False)
    _maybe_update(self)


def end_of_round(self, last_game_state, last_action, events):
    _resolve_bomb(self, events, terminal=True)
    _finish_latest(self, last_game_state, None, events, done=True)
    _maybe_update(self, force=len(self.rollout) >= int(self.model.config["min_update_steps"]))
    self.model.save(MODEL_PATH)


## 6. Checks

These cells test feature shapes, action masks, probability normalization, checkpoint serialization and decision latency. The full and basic masks are also checked using the same feature matrices. A separate framework smoke test follows.


In [ ]:
import importlib, sys, tempfile, time
import numpy as np

sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()

from agent_code.survival_linear_ppo_v4.core import (
    ACTIONS, FEATURE_NAMES, STATE_FEATURE_NAMES, blast_tiles,
    danger_schedule, safe_action_data,
)
from agent_code.survival_linear_ppo_v4.model import LinearPPO

field = np.zeros((17, 17), dtype=int)
field[[0, -1], :] = -1
field[:, [0, -1]] = -1
field[2::2, 2::2] = -1
field[3, 1] = 1
state = {
    "round": 1, "step": 1, "field": field,
    "self": (PPO_AGENT, 0, True, (1, 1)),
    "others": [("opponent", 0, True, (5, 1))],
    "bombs": [], "coins": [(1, 3)],
    "explosion_map": np.zeros_like(field), "user_input": None,
}
context = {"own_bombs": set(), "recent_positions": (), "escape_plan": (), "previous_action": None}
phi, mask, metadata = safe_action_data(state, context)
assert phi.shape == (6, len(FEATURE_NAMES))
assert np.isfinite(phi).all()
assert (4, 1) in blast_tiles(field, (1, 1))
danger, detonations, _ = danger_schedule(state, extra_bomb=(1, 1))
assert detonations[(1, 1)] == 5 and danger[5, 4, 1]
assert mask[ACTIONS.index("BOMB")]

trap_field = -np.ones((7, 7), dtype=int)
trap_field[1, 1] = 0
trap_field[2, 1] = 1
trap_state = {**state, "field": trap_field, "self": (PPO_AGENT, 0, True, (1, 1)), "others": [], "coins": [], "explosion_map": np.zeros_like(trap_field)}
_, trap_mask, _ = safe_action_data(trap_state, context)
assert not trap_mask[ACTIONS.index("BOMB")]

ppo = LinearPPO()
probabilities = ppo.probs(phi, mask)
assert np.isclose(probabilities.sum(), 1.0) and np.all(probabilities[~mask] == 0)

with tempfile.TemporaryDirectory() as temp_dir:
    ppo_path = Path(temp_dir) / "ppo.pkl"
    ppo.actor_opt.t = 7
    ppo.actor_opt.m[:] = 0.25
    ppo.save(ppo_path)
    loaded_ppo = LinearPPO.load(ppo_path)
    assert loaded_ppo.actor_opt.t == 7 and np.allclose(loaded_ppo.actor_opt.m, 0.25)

started = time.perf_counter()
for _ in range(50):
    safe_action_data(state, context)
mean_synthetic_ms = 1000 * (time.perf_counter() - started) / 50
assert mean_synthetic_ms < 250, mean_synthetic_ms
print(f"All PPO V4 unit/serialization checks passed; synthetic shield mean={mean_synthetic_ms:.1f} ms")


# In this sealed pocket, the basic mask allows BOMB while the full mask blocks it.
full_phi, full_mask, _ = safe_action_data(trap_state, {**context, "shield_mode": "full"})
basic_phi, basic_mask, _ = safe_action_data(trap_state, {**context, "shield_mode": "basic"})
assert np.array_equal(full_phi, basic_phi)
assert not full_mask[ACTIONS.index("BOMB")] and basic_mask[ACTIONS.index("BOMB")]
from agent_code.survival_linear_ppo_v4.core import _physical_actions
assert np.all(~basic_mask | _physical_actions(trap_state))
immediate_state = {**state, "bombs": [((1, 3), 0)]}
_, immediate_mask, _ = safe_action_data(immediate_state, {**context, "shield_mode": "basic"})
assert not immediate_mask[ACTIONS.index("WAIT")]
assert immediate_mask.any()
print("Full/basic feature equality and immediate-danger mask checks passed")

In [ ]:
# Compile generated modules before running the framework.
import compileall
for agent_dir in AGENT_DIRS.values():
    assert compileall.compile_dir(agent_dir, quiet=1)
print("All generated agent modules compile")


## 7. Framework smoke test

Fresh runs perform a short two-round training test before the main curriculum. The model is then reset to the saved initial policy. Resume and evaluation-only modes skip this step.


In [ ]:
if CONFIG.mode == "fresh":
    command = [sys.executable, "main.py", "play", "--agents", PPO_AGENT,
               "--train", "1", "--scenario", "coin-heaven", "--n-rounds", "2",
               "--no-gui", "--seed", "991"]
    subprocess.run(command, cwd=REPO_DIR, check=True)
    print("PPO official-framework smoke test passed")
else:
    print("Training smoke test skipped for checkpoint reuse/resume.")

## 8. Experiment utilities

The helper code below runs seeded evaluations, records statistics and saves run metadata such as configuration, source hashes, package versions and checkpoint hashes. Command logs are kept with each evaluation shard.


In [ ]:
%%writefile /content/bomberman_rl/v4_experiment_utils.py
"""Utilities for evaluating saved checkpoints on matched seed blocks."""
from __future__ import annotations
from datetime import datetime, timezone
import ast
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time
import numpy as np
from scipy.stats import t


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def ensure_extended_stats(repo_dir):
    path = Path(repo_dir) / "agents.py"
    source = path.read_text()
    # Locate the statistics mapping through the AST so formatting does not matter.
    tree = ast.parse(source)
    node = next((n.value for n in ast.walk(tree) if isinstance(n, ast.Assign)
                 and any(isinstance(x, ast.Name) and x.id == "EVENT_STAT_MAP" for x in n.targets)), None)
    if not isinstance(node, ast.Dict):
        raise RuntimeError("Cannot locate EVENT_STAT_MAP; inspect framework statistics before proceeding.")
    fields = {ast.unparse(k): ast.literal_eval(v) for k, v in zip(node.keys, node.values)}
    additions = {"e.GOT_KILLED": "deaths", "e.SURVIVED_ROUND": "survived"}
    for key, value in additions.items():
        if key in fields and fields[key] != value:
            raise RuntimeError(f"Unexpected statistics mapping: {key} -> {fields[key]}")
    missing = {k:v for k,v in additions.items() if k not in fields}
    if missing:
        lines = source.splitlines(keepends=True)
        start = sum(map(len, lines[:node.lineno-1])) + node.col_offset
        end = sum(map(len, lines[:node.end_lineno-1])) + node.end_col_offset
        replacement = "{\n" + "".join(f"    {k}: {v!r},\n" for k,v in {**fields, **missing}.items()) + "}"
        source = source[:start] + replacement + source[end:]
    # Initialize all tracked counters so zero-event fields are still written to JSON.
    marker = "# controlled evaluation: explicit zero statistics"
    if marker not in source:
        needle = "self.lifetime_statistics = defaultdict(int)"
        if needle not in source:
            raise RuntimeError("Cannot initialize statistics; inspect Agent.lifetime_statistics")
        source = source.replace(needle,
            'self.lifetime_statistics = defaultdict(int, {key: 0 for key in '
            'set(EVENT_STAT_MAP.values()) | {"rounds", "steps", "time"}})  ' + marker)
    path.write_text(source)


def prepare_opponents(repo_dir):
    """Create seeded copies of the provided rule-based opponents."""
    root = Path(repo_dir) / "agent_code"
    original = root / "rule_based_agent" / "callbacks.py"
    if not original.exists():
        raise FileNotFoundError(original)
    names = []
    for slot in range(3):
        name = f"controlled_rule_{slot}"
        directory = root / name
        directory.mkdir(exist_ok=True)
        (directory / "__init__.py").touch()
        # Copy the complete opponent package.
        for module in original.parent.glob("*.py"):
            shutil.copy2(module, directory / module.name)
        tree = ast.parse(original.read_text())
        class SeedCalls(ast.NodeTransformer):
            def visit_Call(self, node):
                self.generic_visit(node)
                if ast.unparse(node.func) in {"np.random.seed", "numpy.random.seed", "random.seed"}:
                    if not node.args and not node.keywords:
                        node.args = [ast.parse("int(__import__('os').environ['BOMBERMAN_EVAL_SEED']) + " + str(slot), mode="eval").body]
                return node
        tree = ast.fix_missing_locations(SeedCalls().visit(tree))
        source = ast.unparse(tree)
        # Seed before setup as well.
        source += f'''\n\n_controlled_original_setup = setup
def setup(self):
    import os as _os, random as _random, numpy as _np
    _seed = int(_os.environ['BOMBERMAN_EVAL_SEED']) + {slot}
    _random.seed(_seed)
    _np.random.seed(_seed)
    _controlled_original_setup(self)
'''
        (directory / "callbacks.py").write_text(source)
        names.append(name)
    return names


def run_checked(command, cwd, log_path, env=None):
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    started = time.time()
    with path.open("w", encoding="utf-8") as stream:
        result = subprocess.run(command, cwd=cwd, stdout=stream, stderr=subprocess.STDOUT, text=True, env=env)
    if result.returncode:
        tail = "\n".join(path.read_text(errors="replace").splitlines()[-60:])
        raise RuntimeError(f"Command failed ({result.returncode}): {command}\n{tail}")
    return time.time() - started


def source_hashes(repo_dir):
    repo = Path(repo_dir)
    paths = list(repo.glob("*.py"))
    paths += list((repo / "agent_code" / "survival_linear_ppo_v4").glob("*.py"))
    paths += list((repo / "agent_code").glob("controlled_rule_*/*.py"))
    return {str(p.relative_to(repo)): sha256(p) for p in sorted(paths)}


def parse_stats(data, agent_name, expected_rounds, seed, variant, split, elapsed):
    stats = data["by_agent"][agent_name]
    required = {"rounds", "steps", "score", "survived", "deaths", "suicides", "kills", "invalid", "time"}
    if required - stats.keys():
        raise RuntimeError(f"Missing statistics: {sorted(required - stats.keys())}")
    rounds, steps = int(stats["rounds"]), int(stats["steps"])
    if rounds != expected_rounds or steps <= 0:
        raise RuntimeError(f"Incomplete evaluation: expected {expected_rounds} rounds, got {rounds}; steps={steps}")
    opponents = [v for k,v in data["by_agent"].items() if k != agent_name]
    if len(opponents) != 3 or any(int(v["rounds"]) != rounds for v in opponents):
        raise RuntimeError("Expected three rule-based opponents with matching round counts")
    opponent_score = float(np.mean([v["score"] / rounds for v in opponents]))
    result = {"agent": agent_name, "variant": variant, "label": split, "seed": int(seed),
              "rounds": rounds, "steps": steps, "suicides": int(stats["suicides"]),
              "invalid": int(stats["invalid"]), "act_seconds": float(stats["time"]),
              "wall_seconds": elapsed, "opponent_score_per_round": opponent_score}
    for key in ("score", "coins", "kills", "crates", "bombs"):
        result[key + "_per_round"] = float(stats.get(key, 0)) / rounds
    for source, target in (("survived", "survival"), ("deaths", "death"), ("suicides", "suicide")):
        result[target + "_rate"] = float(stats[source]) / rounds
    result["score_advantage"] = result["score_per_round"] - opponent_score
    result["invalid_rate"] = result["invalid"] / steps
    result["mean_act_ms"] = 1000 * result["act_seconds"] / steps
    return result


def evaluate_variant(repo_dir, agent_name, checkpoint, variant, seeds, rounds_per_seed, output_dir, split):
    repo_dir, checkpoint, output_dir = Path(repo_dir), Path(checkpoint), Path(output_dir)
    if not checkpoint.is_file():
        raise FileNotFoundError(f"No checkpoint for {variant}: {checkpoint}")
    if len(set(seeds)) != len(seeds):
        raise ValueError("Duplicate seed blocks")
    output_dir.mkdir(parents=True, exist_ok=True)
    opponents = prepare_opponents(repo_dir)
    model_hash = sha256(checkpoint)
    code_hashes = source_hashes(repo_dir)
    runner = ("import os,random,runpy,sys; import numpy as np; "
              "seed=int(os.environ['BOMBERMAN_EVAL_SEED']); random.seed(seed); np.random.seed(seed); "
              "sys.argv=['main.py']+sys.argv[1:]; runpy.run_path('main.py',run_name='__main__')")
    rows = []
    try:
        for index, seed in enumerate(seeds):
            print(f"  {split}/{variant}: seed {index+1}/{len(seeds)} ({seed})", flush=True)
            shard = output_dir / f"seed_{seed}"
            shard.mkdir(exist_ok=True)
            result_path, request_path = shard / "metrics.json", shard / "request.json"
            request = {"schema": 2, "variant": variant, "split": split, "seed": int(seed),
                       "rounds": rounds_per_seed, "checkpoint_sha256": model_hash,
                       "source_hashes": code_hashes, "opponents": opponents,
                       "scenario": "classic", "action_selection": "greedy", "train": False}
            if request_path.exists() and json.loads(request_path.read_text()) != request:
                raise RuntimeError(f"Changed model/code/protocol in {shard}; use a new run folder.")
            request_path.write_text(json.dumps(request, indent=2))
            if result_path.exists():
                row = json.loads(result_path.read_text())
            else:
                stats_path = shard / "stats.json"
                # Remove stale statistics from an interrupted run.
                stats_path.unlink(missing_ok=True)
                command = [sys.executable, "-c", runner, "play", "--agents", agent_name, *opponents,
                           "--scenario", "classic", "--n-rounds", str(rounds_per_seed),
                           "--no-gui", "--seed", str(seed), "--save-stats", str(stats_path)]
                env = {**os.environ, "BOMBERMAN_EVAL_SEED": str(seed), "PYTHONHASHSEED": str(seed),
                       "BOMBERMAN_EVAL_MODEL_PATH": str(checkpoint.resolve())}
                elapsed = run_checked(command, repo_dir, shard / "run.log", env=env)
                row = parse_stats(json.loads(stats_path.read_text()), agent_name, rounds_per_seed,
                                  seed, variant, split, elapsed)
                if sha256(checkpoint) != model_hash:
                    raise RuntimeError("Evaluation modified weights; discard this shard.")
                temporary = result_path.with_suffix(".tmp")
                temporary.write_text(json.dumps(row, indent=2))
                temporary.replace(result_path)
            rows.append(row)
        assert sha256(checkpoint) == model_hash
    finally:
        # Load evaluation checkpoints through the environment without replacing the training model.
        pass
    return rows


def wilson_upper(successes, trials, z=1.96):
    p = successes / trials
    return float((p + z*z/(2*trials) + z*np.sqrt(p*(1-p)/trials + z*z/(4*trials*trials))) / (1+z*z/trials))


def mean_ci(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or not np.isfinite(values).all():
        raise ValueError("At least two finite seed-block values are required")
    mean = float(values.mean())
    margin = float(t.ppf(.975, len(values)-1) * values.std(ddof=1) / np.sqrt(len(values)))
    return mean, mean-margin, mean+margin


def summarize(rows):
    if not rows or len({r['variant'] for r in rows}) != 1 or len({r['label'] for r in rows}) != 1:
        raise ValueError("Summarize one variant and split at a time")
    if len({r['rounds'] for r in rows}) != 1 or len({r['seed'] for r in rows}) != len(rows):
        raise ValueError("Expected unique, equal-sized seed blocks")
    rounds = sum(r['rounds'] for r in rows)
    steps = sum(r['steps'] for r in rows)
    score, low, high = mean_ci([r['score_per_round'] for r in rows])
    result = {"agent": rows[0]['agent'], "variant": rows[0]['variant'], "label": rows[0]['label'],
              "seeds": len(rows), "rounds": rounds, "score_per_round": score,
              "score_ci_low": low, "score_ci_high": high,
              "suicide_wilson_upper": wilson_upper(sum(r['suicides'] for r in rows), rounds),
              "invalid_rate": sum(r['invalid'] for r in rows)/steps,
              "mean_act_ms": 1000*sum(r['act_seconds'] for r in rows)/steps}
    for key in ("score_advantage", "kills_per_round", "survival_rate", "death_rate", "suicide_rate"):
        result[key] = float(np.mean([r[key] for r in rows]))
    result['selection_objective'] = low - 4.0*result['suicide_wilson_upper']
    return result


def paired_comparison(rows, a, b, question):
    def keyed(name):
        subset = [r for r in rows if r['variant'] == name]
        keys = {(r['label'], r['seed']): r for r in subset}
        if len(keys) != len(subset):
            raise ValueError("Duplicate paired observations")
        return keys
    left, right = keyed(a), keyed(b)
    if set(left) != set(right) or len(left) < 2:
        raise ValueError("Paired comparison requires identical split/seed sets")
    if len({key[0] for key in left}) != 1:
        raise ValueError("Never pool development and test for a paired contrast")
    if any(left[k]['rounds'] != right[k]['rounds'] for k in left):
        raise ValueError("Mismatched rounds in seed block")
    result = []
    for metric in ('score_per_round', 'survival_rate', 'suicide_rate', 'kills_per_round'):
        mean, low, high = mean_ci([left[k][metric] - right[k][metric] for k in sorted(left)])
        result.append({'question': question, 'a': a, 'b': b, 'metric': metric,
                       'paired_seeds': len(left), 'mean_difference': mean, 'ci_low': low, 'ci_high': high,
                       'direction': 'lower_is_better' if metric == 'suicide_rate' else 'higher_is_better'})
    return result


def run_manifest(repo_dir, config, output_path):
    commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
    packages = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True).splitlines()
    manifest = {'created_utc': datetime.now(timezone.utc).isoformat(), 'framework_commit': commit,
                'python': sys.version, 'platform': platform.platform(), 'packages': packages,
                'config': config, 'source_hashes': source_hashes(repo_dir)}
    Path(output_path).write_text(json.dumps(manifest, indent=2))
    return manifest

## 9. Training and checkpoints

Training uses four stages: 20 navigation rounds, 70 bomb-escape rounds, 60 mixed-opponent rounds and 150 combat rounds. This gives a 90-round checkpoint after the first two stages and a 300-round checkpoint after all four.

The notebook saves checkpoint hashes, source snapshots and phase information for reproducibility. Imported checkpoints are checked against the expected feature schema and PPO state. Development results are used for comparison only; the full run exports the 300-round full-mask checkpoint.


In [ ]:
from datetime import datetime, timezone
import json, shutil, sys
import pandas as pd
from v4_experiment_utils import (ensure_extended_stats, prepare_opponents, evaluate_variant,
    run_checked, run_manifest, summarize, paired_comparison, sha256, source_hashes)
from agent_code.survival_linear_ppo_v4.model import LinearPPO

ensure_extended_stats(REPO_DIR)
prepare_opponents(REPO_DIR)
if CONFIG.mode == "resume":
    RUN_DIR = Path(CONFIG.resume_run_dir).expanduser().resolve()
    if not CONFIG.resume_run_dir or not (RUN_DIR / "manifest.json").is_file():
        raise ValueError("resume_run_dir must point to a completed run setup from this revision")
else:
    RUN_DIR = REPO_DIR / "results" / "controlled_ppo" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    RUN_DIR.mkdir(parents=True, exist_ok=False)
RUN_ID = RUN_DIR.name
model_path = AGENT_DIRS[PPO_AGENT] / "model.pkl"
metrics_path = AGENT_DIRS[PPO_AGENT] / "training_metrics.jsonl"
checkpoint_dir = RUN_DIR / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

if CONFIG.mode == "resume":
    manifest = json.loads((RUN_DIR / "manifest.json").read_text())
    current_config = json.loads(json.dumps(asdict(CONFIG)))
    ignored = {"mode", "resume_run_dir", "checkpoint_90", "checkpoint_300"}
    for key, value in manifest['config'].items():
        if key not in ignored and current_config[key] != value:
            raise ValueError(f"Changed config on resume: {key}. Start a new run instead.")
    if manifest['source_hashes'] != source_hashes(REPO_DIR):
        raise ValueError("Source changed since run setup. Resume requires the recorded code.")
else:
    manifest = run_manifest(REPO_DIR, asdict(CONFIG), RUN_DIR / "manifest.json")
    for relative in manifest['source_hashes']:
        target = RUN_DIR / "source_snapshot" / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(REPO_DIR / relative, target)

prior_path = checkpoint_dir / "initial_prior.pkl"
if not prior_path.exists():
    LinearPPO().save(prior_path)
prior = LinearPPO.load(prior_path)
assert prior.n_updates == 0 and prior.actor_opt.t == 0
assert np.array_equal(prior.theta, LinearPPO().theta)

phase_specs = [
    ("ppo_navigation", "coin-heaven", 20, ()),
    ("ppo_bomb_escape", "loot-crate", 70, ()),
    ("ppo_mixed", "classic", 60, ("peaceful_agent", "coin_collector_agent", "rule_based_agent")),
    ("ppo_combat", "classic", 150, ("rule_based_agent", "rule_based_agent", "rule_based_agent")),
]
phase_rows = []
provenance = {"prior_full": {"training_rounds": 0, "origin": "exact V4 constructor; no PPO updates"}}
if CONFIG.mode == "evaluate_existing":
    for label, provided, rounds in ((EARLY, CONFIG.checkpoint_90, 90), (LATE, CONFIG.checkpoint_300, 300)):
        if not provided or not Path(provided).is_file():
            raise FileNotFoundError(f"Provide the actual {rounds}-round checkpoint path: {provided!r}")
        loaded = LinearPPO.load(provided)
        if loaded.n_updates <= 0 or loaded.config.get("shield_mode", "full") != "full":
            raise ValueError(f"{label} must be a trained original full-mask PPO checkpoint")
        if not np.isfinite(loaded.theta).all() or not np.isfinite(loaded.value_w).all():
            raise ValueError("Non-finite checkpoint parameters")
        destination = checkpoint_dir / (label + ".pkl")
        shutil.copy2(provided, destination)
        provenance[label] = {"training_rounds": rounds, "budget_source": "user supplied, not encoded in V4 pickle",
                             "origin": str(Path(provided).resolve()), "n_updates": loaded.n_updates}
    if sha256(checkpoint_dir / (EARLY + ".pkl")) == sha256(checkpoint_dir / (LATE + ".pkl")):
        raise ValueError("The early and late paths contain the same checkpoint")
elif CONFIG.mode == "resume" and (RUN_DIR / "imported_checkpoints.json").exists():
    provenance = json.loads((RUN_DIR / "imported_checkpoints.json").read_text())
else:
    previous = prior_path
    total_rounds = 0
    for index, (name, scenario, normal_rounds, opponents) in enumerate(phase_specs):
        rounds = 1 if CONFIG.profile == "quick" else normal_rounds
        total_rounds += rounds
        checkpoint = checkpoint_dir / f"{index:02d}_{name}.pkl"
        journal = checkpoint_dir / f"{index:02d}_{name}.json"
        metrics_snapshot = checkpoint_dir / f"{index:02d}_{name}_metrics.jsonl"
        if journal.exists():
            record = json.loads(journal.read_text())
            if not checkpoint.exists() or sha256(checkpoint) != record['sha256']:
                raise RuntimeError(f"Missing/modified completed checkpoint: {checkpoint}")
        else:
            # Resume from the last completed phase; partial phases are rerun.
            shutil.copy2(previous, model_path)
            metrics_path.unlink(missing_ok=True)
            if index > 0:
                previous_metrics = checkpoint_dir / f"{index-1:02d}_{phase_specs[index-1][0]}_metrics.jsonl"
                if previous_metrics.exists():
                    shutil.copy2(previous_metrics, metrics_path)
            command = [sys.executable, "main.py", "play", "--agents", PPO_AGENT, *opponents,
                       "--train", "1", "--scenario", scenario, "--n-rounds", str(rounds),
                       "--no-gui", "--seed", str(CONFIG.base_seed + index + 1)]
            print(f"Training {name}: {rounds} rounds (cumulative {total_rounds})", flush=True)
            elapsed = run_checked(command, REPO_DIR, RUN_DIR / "logs" / (name + ".log"))
            trained = LinearPPO.load(model_path)
            shutil.copy2(model_path, checkpoint)
            if metrics_path.exists():
                shutil.copy2(metrics_path, metrics_snapshot)
            record = {"phase": name, "scenario": scenario, "rounds": rounds,
                      "cumulative_rounds": total_rounds, "opponents": opponents,
                      "seed": CONFIG.base_seed + index + 1, "minutes": elapsed/60,
                      "checkpoint": str(checkpoint), "sha256": sha256(checkpoint),
                      "n_updates": trained.n_updates}
            temporary = journal.with_suffix(".tmp")
            temporary.write_text(json.dumps(record, indent=2))
            temporary.replace(journal)  # Complete only after the checkpoint and metrics exist.
        phase_rows.append(record)
        pd.DataFrame(phase_rows).to_csv(RUN_DIR / "phase_log.csv", index=False)
        previous = checkpoint
        if index in (1, 3):
            label = EARLY if index == 1 else LATE
            shutil.copy2(checkpoint, checkpoint_dir / (label + ".pkl"))
            provenance[label] = {"training_rounds": total_rounds, "origin": name,
                                 "n_updates": record['n_updates'], "budget_source": "phase journal"}

if CONFIG.mode == "evaluate_existing":
    (RUN_DIR / "imported_checkpoints.json").write_text(json.dumps(provenance, indent=2))
VARIANTS = {"prior_full": prior_path,
            EARLY: checkpoint_dir / (EARLY + ".pkl"), LATE: checkpoint_dir / (LATE + ".pkl")}
for label, path in VARIANTS.items():
    model = LinearPPO.load(path)
    provenance[label].update({"checkpoint": str(path), "sha256": sha256(path), "n_updates": model.n_updates})
(RUN_DIR / "checkpoint_provenance.json").write_text(json.dumps(provenance, indent=2))

development_seed_rows = []
for variant in (EARLY, LATE):
    development_seed_rows.extend(evaluate_variant(REPO_DIR, PPO_AGENT, VARIANTS[variant], variant,
        DEV_SEEDS, DEV_ROUNDS, RUN_DIR / "development" / variant, "development"))
pd.DataFrame(development_seed_rows).to_csv(RUN_DIR / "development_by_seed.csv", index=False)
development = pd.DataFrame([{
    **summarize([row for row in development_seed_rows if row['variant'] == variant]),
    "phase": variant, "checkpoint": str(VARIANTS[variant])} for variant in (EARLY, LATE)])
development.to_csv(RUN_DIR / "development_summary.csv", index=False)
pd.DataFrame(paired_comparison(development_seed_rows, LATE, EARLY, "Later curriculum stages")).to_csv(
    RUN_DIR / "paired_development_differences.csv", index=False)
display(development)
# Build the basic-mask variant from the late checkpoint.
basic = LinearPPO.load(VARIANTS[LATE])
basic.config['shield_mode'] = 'basic'
basic_path = checkpoint_dir / "late_basic.pkl"
basic.save(basic_path)
full = LinearPPO.load(VARIANTS[LATE])
assert np.array_equal(basic.theta, full.theta) and np.array_equal(basic.value_w, full.value_w)
VARIANTS['late_basic'] = basic_path
provenance['late_basic'] = {**provenance[LATE], "origin": LATE,
    "checkpoint": str(basic_path), "sha256": sha256(basic_path),
    "change": "Only config.shield_mode=basic; same actor/critic/optimizer weights"}
(RUN_DIR / "checkpoint_provenance.json").write_text(json.dumps(provenance, indent=2))
shutil.copy2(VARIANTS[LATE], model_path)
print("Checkpoint used for late-policy analysis:", LATE)
print("Run folder (use this to resume):", RUN_DIR)

## 10. Training diagnostics

Shaped reward is useful for checking PPO optimization. Final performance is evaluated separately with held-out score and safety metrics.


In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

path = AGENT_DIRS[PPO_AGENT] / "training_metrics.jsonl"
if not path.exists() or not path.read_text().strip():
    print("No PPO diagnostics yet")
else:
    frame = pd.DataFrame(json.loads(line) for line in path.read_text().splitlines())
    display(frame.tail())
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=frame, x="update", y="reward_mean", marker="o")
    plt.title("Linear PPO: mean shaped reward by update")
    plt.grid(alpha=.25)
    plt.show()

if not development.empty:
    plt.figure(figsize=(10, 4))
    sns.scatterplot(data=development, x="suicide_rate", y="score_per_round", s=110)
    for _, row in development.iterrows():
        plt.annotate(row["phase"], (row["suicide_rate"], row["score_per_round"]), xytext=(5, 5), textcoords="offset points")
    plt.axhline(CONFIG.v3_score_baseline, color="black", linestyle="--", label="Historical V3 score (unmatched)")
    plt.axvline(CONFIG.max_suicide_rate, color="crimson", linestyle="--", label="suicide gate")
    plt.title("PPO development checkpoints: score-safety frontier")
    plt.grid(alpha=.25)
    plt.legend()
    plt.show()

## 11. Test evaluation

This section evaluates four variants on the same held-out seed blocks: the initial policy, the 90-round checkpoint, the 300-round checkpoint and the 300-round policy with the basic mask. All evaluations use greedy inference without training, and checkpoint files are checked after each run.


In [ ]:
from v4_experiment_utils import evaluate_variant, summarize, paired_comparison

test_rows = []
try:
    for variant in ["prior_full", EARLY, LATE, "late_basic"]:
        print("Test variant:", variant, flush=True)
        test_rows.extend(evaluate_variant(REPO_DIR, PPO_AGENT, VARIANTS[variant], variant,
                         TEST_SEEDS, TEST_ROUNDS, RUN_DIR / "test" / variant, "test"))
finally:
    shutil.copy2(VARIANTS[LATE], AGENT_DIRS[PPO_AGENT] / "model.pkl")
test_by_seed = pd.DataFrame(test_rows)
test_summary = pd.DataFrame([summarize(group.to_dict("records"))
                            for _, group in test_by_seed.groupby("variant", sort=False)])
test_by_seed.to_csv(RUN_DIR / "test_by_seed.csv", index=False)
test_summary.to_csv(RUN_DIR / "test_summary.csv", index=False)
contrasts = [(EARLY, "prior_full", "PPO updates after 90 rounds"),
             (LATE, EARLY, "Additional mixed/combat training"),
             (LATE, "late_basic", "Full versus basic action mask")]
paired_results = pd.DataFrame([row for a, b, question in contrasts
                              for row in paired_comparison(test_rows, a, b, question)])
paired_results.to_csv(RUN_DIR / "paired_test_differences.csv", index=False)
display(test_summary)
display(paired_results)
# Late-checkpoint metrics are the 300-round results in the full profile.
final = test_by_seed[test_by_seed.variant == LATE].copy()
final_summary = test_summary[test_summary.variant == LATE].copy()
final.to_csv(RUN_DIR / "final_by_seed.csv", index=False)
final_summary.to_csv(RUN_DIR / "final_summary.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for i, row in test_summary.reset_index(drop=True).iterrows():
    axes[0].errorbar(i, row.score_per_round,
        yerr=[[row.score_per_round-row.score_ci_low], [row.score_ci_high-row.score_per_round]],
        fmt="o", capsize=4, color="navy")
    points = test_by_seed[test_by_seed.variant == row.variant]
    axes[1].scatter(points.suicide_rate, points.score_per_round, label=row.variant, alpha=.7)
axes[0].set_xticks(range(len(test_summary)), test_summary.variant, rotation=20, ha="right")
axes[0].set_ylabel("Score / round (95% CI across seed blocks)")
axes[0].set_title("Matched held-out comparisons")
axes[1].set(xlabel="Suicide rate per seed block", ylabel="Score / round")
axes[1].legend(fontsize=8)
for axis in axes: axis.grid(alpha=.25)
fig.savefig(RUN_DIR / "test_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

score_differences = paired_results[paired_results.metric == "score_per_round"]
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for i, row in enumerate(score_differences.itertuples()):
    ax.errorbar(row.mean_difference, i,
                xerr=[[row.mean_difference-row.ci_low], [row.ci_high-row.mean_difference]],
                fmt="o", capsize=4)
ax.set_yticks(range(len(score_differences)), score_differences.question)
ax.axvline(0, color="gray", linestyle="--")
ax.set_xlabel("Paired score difference: A − B (95% seed-block t interval)")
fig.savefig(RUN_DIR / "paired_score_differences.png", dpi=180, bbox_inches="tight")
plt.show()

## 12. Final checks

The final evaluation is checked against the score, suicide-rate, invalid-action and latency thresholds defined in the configuration. The results are recorded in the exported manifest.


In [ ]:
gate_table = final_summary.copy()
gate_table["score_gate"] = gate_table.score_per_round >= CONFIG.v3_score_baseline
gate_table["suicide_gate"] = gate_table.suicide_rate <= CONFIG.max_suicide_rate
gate_table["invalid_gate"] = gate_table.invalid_rate <= CONFIG.max_invalid_rate
gate_table["latency_gate"] = gate_table.mean_act_ms <= CONFIG.max_mean_act_ms
gate_columns = ["score_gate", "suicide_gate", "invalid_gate", "latency_gate"]
gate_table["passes_all"] = gate_table[gate_columns].all(axis=1)
display(gate_table[["agent", "score_per_round", "suicide_rate", "survival_rate", "mean_act_ms", *gate_columns, "passes_all"]])

PPO_PASSES_ALL_GATES = bool(gate_table.iloc[0]["passes_all"])
if CONFIG.profile == "quick":
    print("Quick profile complete. Use the full profile for final metrics.")
elif PPO_PASSES_ALL_GATES:
    print("All configured checks passed.")
else:
    failed = [column for column in gate_columns if not bool(gate_table.iloc[0][column])]
    print("Some configured checks did not pass.")
    print("Failed gates:", failed)
    print("The archive will still be created and the manifest will record the result.")


## 13. Model inspection

Because the actor is linear, its learned coefficients can be compared directly with the initial handcrafted weights. The cell also lists the seed blocks with the weakest evaluation results for debugging.


In [ ]:
from agent_code.survival_linear_ppo_v4.core import FEATURE_NAMES
from agent_code.survival_linear_ppo_v4.model import LinearPPO

initial_theta = LinearPPO.load(VARIANTS["prior_full"]).theta
linear_weights = pd.DataFrame({"feature": FEATURE_NAMES, "initial_prior": initial_theta})
for variant in (EARLY, LATE):
    weights = LinearPPO.load(VARIANTS[variant]).theta
    linear_weights[variant] = weights
    linear_weights[variant + "_delta"] = weights - initial_theta
    linear_weights[variant + "_unchanged"] = np.isclose(weights, initial_theta, atol=1e-12, rtol=0)
linear_weights.to_csv(RUN_DIR / "actor_weight_changes.csv", index=False)
display(linear_weights)
failure_seeds = final.sort_values(["suicide_rate", "score_per_round"], ascending=[False, True]).head(5)
display(failure_seeds[["seed", "score_per_round", "suicide_rate", "death_rate", "mean_act_ms"]])
print("Coefficient changes show how the linear policy moved from its initial weights.")

## 14. Export

The full profile exports the `ppo_300_full` agent. The model hash is checked against the recorded checkpoint before packaging, and `MANIFEST.json` stores the training budget, evaluation metrics and final checks.


In [ ]:
import hashlib, json, shutil, zipfile
from pathlib import Path

# Export the 300-round full-mask checkpoint.
if CONFIG.profile != "ppo300":
    raise RuntimeError("Final export requires profile='ppo300'; quick mode is analysis only.")
final_checkpoint = VARIANTS["ppo_300_full"]
final_provenance = provenance["ppo_300_full"]
if final_provenance["training_rounds"] != 300:
    raise RuntimeError("Final checkpoint must have 300 training rounds in its provenance.")
final_checkpoint_sha256 = sha256(final_checkpoint)
if final_checkpoint_sha256 != final_provenance["sha256"]:
    raise RuntimeError("The 300-round checkpoint changed after provenance was recorded.")
if LinearPPO.load(final_checkpoint).config.get("shield_mode", "full") != "full":
    raise RuntimeError("Final export requires the original full-mask checkpoint.")
if len(gate_table) != 1 or gate_table.iloc[0]["variant"] != "ppo_300_full":
    raise RuntimeError("Run the analysis for ppo_300_full before exporting its metrics.")

export_root = Path("/content/v4_ppo_export")
if export_root.exists():
    shutil.rmtree(export_root)
export_agent = export_root / PPO_AGENT
export_agent.mkdir(parents=True)

required = ["__init__.py", "callbacks.py", "core.py", "model.py", "model.pkl"]
for filename in required:
    source_path = final_checkpoint if filename == "model.pkl" else AGENT_DIRS[PPO_AGENT] / filename
    if not source_path.exists():
        raise FileNotFoundError(source_path)
    shutil.copy2(source_path, export_agent / filename)

evaluation = gate_table.iloc[0].to_dict()
manifest = {
    "agent": PPO_AGENT,
    "non_neural": True,
    "final_variant": "ppo_300_full",
    "training_rounds": 300,
    "checkpoint_sha256": final_checkpoint_sha256,
    "checkpoint_provenance": final_provenance,
    "integration_only": False,
    "status": "passed_all_gates" if PPO_PASSES_ALL_GATES else "did_not_pass_all_gates",
    "passes_all_gates": PPO_PASSES_ALL_GATES,
    "evaluation": evaluation,
    "files": {},
}
for path in sorted(export_agent.iterdir()):
    if path.is_file():
        manifest["files"][path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
(export_agent / "MANIFEST.json").write_text(json.dumps(manifest, indent=2, default=str))

zip_path = Path(shutil.make_archive("/content/final-project-ppo-v4", "zip", export_root))
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
    assert archive.testzip() is None
    assert all(name.startswith(PPO_AGENT + "/") for name in names)
    assert f"{PPO_AGENT}/model.pkl" in names
    if hashlib.sha256(archive.read(f"{PPO_AGENT}/model.pkl")).hexdigest() != final_checkpoint_sha256:
        raise RuntimeError("Packaged model does not match the 300-round checkpoint.")

if not PPO_PASSES_ALL_GATES:
    print("WARNING: PPO failed one or more gates; see MANIFEST.json before using this archive.")
print("Verified 300-round PPO archive:", zip_path, names)

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download manually:", zip_path)

In [ ]:
# Package checkpoints, tables, logs and source snapshots.
report_lines = [
    "# Bomberman experiment summary", "",
    "Final agent: ppo_300_full (300 training rounds).",
    f"Development: {len(DEV_SEEDS)*DEV_ROUNDS} games/checkpoint; test: {len(TEST_SEEDS)*TEST_ROUNDS} games/variant.",
    "", "Differences below are A minus B. Rate differences are percentage points.",
    "Intervals are paired 95% t intervals across matched seed blocks.", ""]
for row in paired_results.itertuples():
    scale = 100 if row.metric.endswith("_rate") else 1
    units = " percentage points" if scale == 100 else " per round"
    report_lines.append(f"- {row.a} minus {row.b}, {row.metric}: "
        f"{scale*row.mean_difference:.3f}{units}; 95% CI "
        f"[{scale*row.ci_low:.3f}, {scale*row.ci_high:.3f}].")
report_lines.extend(["", "Notes:",
    "- Results come from one trained checkpoint sequence.",
    "- The 90-to-300 comparison combines the mixed and combat stages.",
    "- The basic-mask run keeps the same learned weights and features.",
    "- Historical V3 scores were not measured with this matched protocol."])
if CONFIG.profile == "quick":
    report_lines.insert(2, "Quick integration run only.")
(RUN_DIR / "report_findings.md").write_text("\n".join(report_lines))
shutil.copy2(zip_path, RUN_DIR / zip_path.name)
evidence_zip = Path(shutil.make_archive(str(REPO_DIR.parent / (RUN_ID + "_evidence")), "zip", RUN_DIR))
with zipfile.ZipFile(evidence_zip) as archive:
    assert archive.testzip() is None
print("Evidence archive:", evidence_zip)
try:
    from google.colab import files
    files.download(str(evidence_zip))
except ImportError:
    print("Download manually:", evidence_zip)

## 15. Report outputs

The main files generated for the report are:

- `development_summary.csv` and `paired_development_differences.csv` for the 90-round vs 300-round comparison;
- `test_summary.csv` and `paired_test_differences.csv` for the held-out variants;
- `actor_weight_changes.csv` for changes in the linear policy weights;
- the saved figures and logs in the evidence ZIP.

The full-mask/basic-mask comparison uses the same 300-round policy weights, so it measures the effect of changing the inference-time mask rather than retraining a separate policy.


## References

- Official project framework and API: https://github.com/ukoethe/bomberman_rl
- Reference engineering repository: https://github.com/dkalsan/bomberman_rl
- Reference report: `MLE_BombermanProject (1).pdf`
- Assignment: `final_project (1)(20260903-072715).pdf`

Check the relevant license and citation requirements for any material reused in the final report.
